# C2 — Dự báo Màu sắc & SKU
### Câu hỏi 2: Màu sắc nào sẽ được ưa chuộng? | DATA EXPLORERS 2026

**Mục tiêu:**
- Màu sắc nào tăng nhu cầu theo mùa trong Q2/2026
- Tỷ trọng cơ cấu màu sắc dự kiến Q2/2026
- SKU nào có nguy cơ bán chậm

> Chạy **C0** trước để sinh `data_raw/fact_sales_full.csv`

# Thư viện và dữ liệu

In [ ]:
# thư viện và dữ liệu
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
import pmdarima as pm

# tải dữ liệu giao dịch đầy đủ (SQL + PDF T3/2026)
df_fact = pd.read_csv('data_raw/fact_sales_full.csv', parse_dates=['order_date'])
df_fact.head()

In [ ]:
# tổng hợp theo màu sắc × tháng
df_color = (df_fact
    .groupby([df_fact['order_date'].dt.to_period('M'), 'color'])
    .agg(qty=('quantity','sum'), revenue=('line_total','sum'))
    .reset_index()
)
df_color['ds'] = df_color['order_date'].dt.to_timestamp()
df_color = df_color.drop(columns=['order_date'])
df_color.head()

In [ ]:
# đổi tên biến
df_color = df_color.rename(columns={'qty': 'y'})
df_color.head(1)

In [ ]:
# biến ngày
df_color['ds'] = pd.to_datetime(df_color['ds'])
print(f"Màu sắc có mặt: {df_color['color'].nunique()}")
df_color['color'].value_counts().head(10)

In [ ]:
# heatmap màu sắc theo tháng
pivot = df_color.pivot_table(index='ds', columns='color', values='y', fill_value=0)
fig, ax = plt.subplots(figsize=(16, 6))
im = ax.imshow(pivot.T, aspect='auto', cmap='YlOrRd')
ax.set_yticks(range(len(pivot.columns)))
ax.set_yticklabels(pivot.columns, fontsize=8)
ax.set_xticks(range(len(pivot.index)))
ax.set_xticklabels([str(d.date()) for d in pivot.index], rotation=45, fontsize=8)
ax.set_title('Heatmap Số lượng theo Màu sắc × Tháng')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show();

# Ngày lễ

In [ ]:
# Tết Nguyên Đán — ảnh hưởng màu đỏ/vàng
dates_tet = pd.to_datetime(['2025-01-29', '2026-01-17'])
tet = pd.DataFrame({"holiday":      "tet_nguyen_dan",
                    "ds":           dates_tet,
                    "lower_window": -7,
                    "upper_window": 7})

In [ ]:
# Ngày Thiếu nhi 1/6 — ảnh hưởng màu sắc tươi sáng KIDBIKE
dates_thieu_nhi = pd.to_datetime(['2025-06-01', '2026-06-01'])
thieu_nhi = pd.DataFrame({"holiday":      "ngay_thieu_nhi",
                           "ds":           dates_thieu_nhi,
                           "lower_window": -14,
                           "upper_window": 3})

In [ ]:
# gộp ngày lễ
holidays = pd.concat([tet, thieu_nhi])
holidays

In [ ]:
# tách tập train và tập dự báo
training  = df_color[df_color['ds'] <  '2026-04-01']
future_ds = pd.DataFrame({'ds': pd.date_range('2026-04-01', periods=3, freq='MS')})

In [ ]:
# lấy tham số tốt nhất
parameters = pd.read_csv('Forecasting Product/best_params_prophet.csv', index_col=0)
changepoint_prior_scale = float(parameters.loc['changepoint_prior_scale'][0])
holidays_prior_scale    = float(parameters.loc['holidays_prior_scale'][0])
seasonality_prior_scale = float(parameters.loc['seasonality_prior_scale'][0])
seasonality_mode        = parameters.loc['seasonality_mode'][0]

In [ ]:
# trích xuất giá trị tham số
print(f"seasonality_mode        : {seasonality_mode}")
print(f"changepoint_prior_scale : {changepoint_prior_scale}")

In [ ]:
# mô hình Prophet — vòng lặp qua từng màu sắc chủ đạo
top_colors = df_color.groupby('color')['y'].sum().nlargest(8).index.tolist()
forecasts_color = {}

for color in top_colors:
    df_c = training[training['color'] == color][['ds','y']].copy()
    if len(df_c) < 3:
        continue
    m = Prophet(holidays=holidays,
                seasonality_mode=seasonality_mode,
                seasonality_prior_scale=seasonality_prior_scale,
                holidays_prior_scale=holidays_prior_scale,
                changepoint_prior_scale=changepoint_prior_scale)
    m.fit(df_c)
    fc = m.predict(pd.concat([df_c[['ds']], future_ds]).reset_index(drop=True))
    forecasts_color[color] = fc[['ds','yhat']].tail(3)
    print(f"  ✅ {color}")

print("Xong vòng lặp màu sắc")

# Dự báo

In [ ]:
# tạo dataframe tương lai cho SARIMAX
params_s = pd.read_csv('Forecasting Product/best_params_sarimax.csv', index_col=0)
p = int(params_s.loc['p'][0]); d = int(params_s.loc['d'][0]); q = int(params_s.loc['q'][0])
P = int(params_s.loc['P'][0]); D = int(params_s.loc['D'][0]); Q = int(params_s.loc['Q'][0])

In [ ]:
# dự báo SARIMAX cho màu bán chạy nhất
top1_color = top_colors[0]
df_top1 = training[training['color'] == top1_color][['ds','y']].set_index('ds').sort_index()
model_s = pm.ARIMA(order=(p,d,q), seasonal_order=(P,D,Q,12),
                   suppress_warnings=True, enforce_stationarity=False)
model_s.fit(df_top1['y'])
pred_s = model_s.predict(n_periods=3)
print(f"SARIMAX dự báo màu '{top1_color}' — Q2/2026:", pred_s)

In [ ]:
# trực quan hóa xu hướng màu sắc
fig, ax = plt.subplots(figsize=(14, 5))
for color, fc in forecasts_color.items():
    hist = training[training['color']==color].groupby('ds')['y'].sum()
    ax.plot(hist.index, hist.values, alpha=0.6, label=color)
    ax.plot(fc['ds'], fc['yhat'], linestyle='--', marker='o', alpha=0.8)
ax.set_title('Xu hướng màu sắc — Lịch sử + Dự báo Q2/2026')
ax.set_ylabel('Số lượng')
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.show();

# Cơ cấu màu sắc Q2/2026

In [ ]:
# tỷ trọng màu sắc dự kiến Q2/2026
color_q2 = {c: fc['yhat'].sum() for c, fc in forecasts_color.items()}
df_mix = pd.Series(color_q2).sort_values(ascending=False)
df_mix_pct = df_mix / df_mix.sum() * 100

fig, ax = plt.subplots(figsize=(10, 5))
df_mix_pct.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Tỷ trọng màu sắc dự kiến Q2/2026 (%)')
ax.set_ylabel('%')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show();

# Phát hiện SKU bán chậm

In [ ]:
# tổng hợp theo SKU × tháng
df_sku = (df_fact
    .groupby([df_fact['order_date'].dt.to_period('M'), 'product_code', 'product_name', 'color'])
    .agg(qty=('quantity','sum'))
    .reset_index()
)
df_sku['ds'] = df_sku['order_date'].dt.to_timestamp()

In [ ]:
# tính tốc độ tăng trưởng 3 tháng gần nhất per SKU
last3 = df_sku[df_sku['ds'] >= '2025-12-01']
growth = (last3.groupby('product_code')['qty']
          .apply(lambda x: x.pct_change().mean())
          .reset_index()
          .rename(columns={'qty':'growth_rate'}))
slow = growth[growth['growth_rate'] < -0.15].merge(
    df_fact[['product_code','product_name','color']].drop_duplicates(), on='product_code')
slow = slow.sort_values('growth_rate').head(20)
print(f"SKU nguy cơ bán chậm: {len(slow)}")
slow

In [ ]:
# biểu đồ SKU bán chậm
fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(slow['product_name'].str[:30], slow['growth_rate']*100, color='tomato')
ax.set_title('Top SKU nguy cơ bán chậm (Tăng trưởng âm)')
ax.set_xlabel('Tốc độ tăng trưởng (%)')
plt.tight_layout()
plt.show();

# Xuất kết quả

In [ ]:
# gộp dự báo màu thành 1 DataFrame
rows = []
for color, fc in forecasts_color.items():
    fc = fc.copy(); fc['color'] = color
    rows.append(fc)
df_color_forecast = pd.concat(rows, ignore_index=True)
df_color_forecast

In [ ]:
# xuất kết quả
df_color_forecast.to_csv('Forecasting Product/Ensemble/predictions_sarimax.csv', index=False)
slow.to_csv('data_raw/slow_moving_skus.csv', index=False)
print('✅ Đã lưu predictions màu + danh sách SKU bán chậm')